In [2]:
import torch
print(f"get device name : {torch.cuda.get_device_name(0)}")
print(f"is available : {torch.cuda.is_available()}")
print(f"torch version : {torch.__version__}")

get device name : NVIDIA GeForce RTX 3060 Ti
is available : True
torch version : 2.7.1+cu126


# 성능 비교 결과

실제 MobileNetV2 모델에 대한 양자화 적용 결과:

- 원본 모델 크기: 8.9MB
- 양자화 후 모델 크기: 2.35MB (약 4배 감소)
- Top-1 및 Top-5 정확도: 원본과 거의 동일한 성능 유지

In [1]:
import torch
import torch.nn as nn
import torch.quantization
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models.quantization import mobilenet_v2

# 1. MobileNetV2 모델 로드 및 준비
model = mobilenet_v2(pretrained=True, quantize=False)
model.eval()
model.fuse_model() 

model.qconfig = torch.quantization.default_qconfig  # 기본 양자화 설정 (float32 -> int8)
torch.quantization.prepare(model, inplace=True)

C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


QuantizableMobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): ConvReLU2d(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
        (1): ReLU()
        (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
      )
      (1): Identity()
      (2): Identity()
    )
    (1): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): ConvReLU2d(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32)
            (1): ReLU()
            (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
          )
          (1): Identity()
          (2): Identity()
        )
        (1): Conv2d(
          32, 16, kernel_size=(1, 1), stride=(1, 1)
          (activation_post_process): MinMaxObserver(min_val=inf, max_val=-inf)
        )
        (2): Identity()
      )
      (skip_add): FloatFunctional(
        (activation_post_process): 

In [2]:
# 2. 데이터 로드 및 전처리
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

calibration_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
calibration_dataloader = DataLoader(calibration_data, batch_size=32, shuffle=True)

In [3]:
# 3. 보정 데이터로 양자화 파라미터 결정
with torch.no_grad():
    for i, batch in enumerate(calibration_dataloader):
        print(f"sequence : {i}")
        inputs, labels = batch
        model(inputs)

sequence : 0
sequence : 1
sequence : 2
sequence : 3
sequence : 4
sequence : 5
sequence : 6
sequence : 7
sequence : 8
sequence : 9
sequence : 10
sequence : 11
sequence : 12
sequence : 13
sequence : 14
sequence : 15
sequence : 16
sequence : 17
sequence : 18
sequence : 19
sequence : 20
sequence : 21
sequence : 22
sequence : 23
sequence : 24
sequence : 25
sequence : 26
sequence : 27
sequence : 28
sequence : 29
sequence : 30
sequence : 31
sequence : 32
sequence : 33
sequence : 34
sequence : 35
sequence : 36
sequence : 37
sequence : 38
sequence : 39
sequence : 40
sequence : 41
sequence : 42
sequence : 43
sequence : 44
sequence : 45
sequence : 46
sequence : 47
sequence : 48
sequence : 49
sequence : 50
sequence : 51
sequence : 52
sequence : 53
sequence : 54
sequence : 55
sequence : 56
sequence : 57
sequence : 58
sequence : 59
sequence : 60
sequence : 61
sequence : 62
sequence : 63
sequence : 64
sequence : 65
sequence : 66
sequence : 67
sequence : 68
sequence : 69
sequence : 70
sequence : 71
se

sequence : 554
sequence : 555
sequence : 556
sequence : 557
sequence : 558
sequence : 559
sequence : 560
sequence : 561
sequence : 562
sequence : 563
sequence : 564
sequence : 565
sequence : 566
sequence : 567
sequence : 568
sequence : 569
sequence : 570
sequence : 571
sequence : 572
sequence : 573
sequence : 574
sequence : 575
sequence : 576
sequence : 577
sequence : 578
sequence : 579
sequence : 580
sequence : 581
sequence : 582
sequence : 583
sequence : 584
sequence : 585
sequence : 586
sequence : 587
sequence : 588
sequence : 589
sequence : 590
sequence : 591
sequence : 592
sequence : 593
sequence : 594
sequence : 595
sequence : 596
sequence : 597
sequence : 598
sequence : 599
sequence : 600
sequence : 601
sequence : 602
sequence : 603
sequence : 604
sequence : 605
sequence : 606
sequence : 607
sequence : 608
sequence : 609
sequence : 610
sequence : 611
sequence : 612
sequence : 613
sequence : 614
sequence : 615
sequence : 616
sequence : 617
sequence : 618
sequence : 619
sequence :

sequence : 1094
sequence : 1095
sequence : 1096
sequence : 1097
sequence : 1098
sequence : 1099
sequence : 1100
sequence : 1101
sequence : 1102
sequence : 1103
sequence : 1104
sequence : 1105
sequence : 1106
sequence : 1107
sequence : 1108
sequence : 1109
sequence : 1110
sequence : 1111
sequence : 1112
sequence : 1113
sequence : 1114
sequence : 1115
sequence : 1116
sequence : 1117
sequence : 1118
sequence : 1119
sequence : 1120
sequence : 1121
sequence : 1122
sequence : 1123
sequence : 1124
sequence : 1125
sequence : 1126
sequence : 1127
sequence : 1128
sequence : 1129
sequence : 1130
sequence : 1131
sequence : 1132
sequence : 1133
sequence : 1134
sequence : 1135
sequence : 1136
sequence : 1137
sequence : 1138
sequence : 1139
sequence : 1140
sequence : 1141
sequence : 1142
sequence : 1143
sequence : 1144
sequence : 1145
sequence : 1146
sequence : 1147
sequence : 1148
sequence : 1149
sequence : 1150
sequence : 1151
sequence : 1152
sequence : 1153
sequence : 1154
sequence : 1155
sequence

In [4]:
quantized_model = torch.quantization.convert(model, inplace=True)  # 모델을 양자화된 모델로 변환
# 5. 변환된 양자화 모델을 확인
print(quantized_model)

QuantizableMobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): QuantizedConvReLU2d(3, 32, kernel_size=(3, 3), stride=(2, 2), scale=0.04931391775608063, zero_point=0, padding=(1, 1))
      (1): Identity()
      (2): Identity()
    )
    (1): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): QuantizedConvReLU2d(32, 32, kernel_size=(3, 3), stride=(1, 1), scale=0.10223246365785599, zero_point=0, padding=(1, 1), groups=32)
          (1): Identity()
          (2): Identity()
        )
        (1): QuantizedConv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), scale=0.13919185101985931, zero_point=76)
        (2): Identity()
      )
      (skip_add): QFunctional(
        scale=1.0, zero_point=0
        (activation_post_process): Identity()
      )
    )
    (2): QuantizableInvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): QuantizedConvReLU2d(16, 96, kernel_size=(1, 1), s

C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\ao\quantization\utils.py:408: UserWarning: must run observer before calling calculate_qparams. Returning default values.
  warnings.warn(


In [6]:
def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    accuracy = 100 * correct / total
    return accuracy

# 양자화 모델 성능 평가
accuracy = evaluate(quantized_model, calibration_dataloader)
print(f"Accuracy of the quantized model: {accuracy:.2f}%")

Accuracy of the quantized model: 0.06%


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.quantization
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models.quantization import mobilenet_v2

# 1. 데이터 로드 및 전처리
data_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=data_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=data_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. 평가 함수 정의
def evaluate(model, dataloader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100. * correct / total

# 3. 기본 모델 정의 및 초기 정확도 측정
model_fp32 = mobilenet_v2(pretrained=True, quantize=False)
model_fp32.classifier[1] = nn.Linear(model_fp32.last_channel, 10)
model_fp32.eval()

print("[Original Model] Accuracy before tuning:")
print(f"Accuracy: {evaluate(model_fp32, test_loader):.2f}%")

# ========== PTQ ==========
ptq_model = mobilenet_v2(pretrained=True, quantize=False)
ptq_model.classifier[1] = nn.Linear(ptq_model.last_channel, 10)
ptq_model.eval()
ptq_model.fuse_model()
ptq_model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(ptq_model, inplace=True)

with torch.no_grad():
    for inputs, _ in train_loader:
        ptq_model(inputs)

ptq_model = torch.quantization.convert(ptq_model, inplace=False)
print("\n[PTQ Model] Accuracy:")
print(f"Accuracy: {evaluate(ptq_model, test_loader):.2f}%")

# ========== QAT ==========
qat_model = mobilenet_v2(pretrained=True, quantize=False)
qat_model.classifier[1] = nn.Linear(qat_model.last_channel, 10)
qat_model.train()
qat_model.fuse_model()
qat_model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
torch.quantization.prepare_qat(qat_model, inplace=True)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(qat_model.parameters(), lr=0.001, momentum=0.9)

print("\n[QAT Model] Training...")
for epoch in range(5):  # 짧게 튜닝
    qat_model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = qat_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

qat_model.eval()
qat_model = torch.quantization.convert(qat_model, inplace=False)
print("\n[QAT Model] Accuracy after tuning:")
print(f"Accuracy: {evaluate(qat_model, test_loader):.2f}%")

[Original Model] Accuracy before tuning:
Accuracy: 9.37%


C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\ao\quantization\observer.py:244: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\ao\quantization\observer.py:1333: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  warnings.warn(



[PTQ Model] Accuracy:
Accuracy: 8.30%

[QAT Model] Training...
Epoch 1, Loss: 0.5900
Epoch 2, Loss: 0.2742
Epoch 3, Loss: 0.2006
Epoch 4, Loss: 0.1521
Epoch 5, Loss: 0.1216

[QAT Model] Accuracy after tuning:
Accuracy: 93.63%


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.quantization
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from torchvision.models.quantization import mobilenet_v2

device = torch.device("cpu")

if(torch.cuda.is_available()) :
    print("torce device : cuda")
    device = torch.device("cuda")
else:
    print("torch device : cpu")


# 1. 데이터 로드 및 전처리
data_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=data_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=data_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# 2. 평가 함수 정의
def evaluate(model, dataloader):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100. * correct / total

# 3. 기본 모델 정의 및 전체 학습 (baseline 정확도 향상)
model_fp32 = mobilenet_v2(pretrained=True, quantize=False)
model_fp32.classifier[1] = nn.Linear(model_fp32.last_channel, 10)
model_fp32.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model_fp32.parameters(), lr=0.001, momentum=0.9)

print("\n[Original Model] Training full model...")
model_fp32.train()
for epoch in range(10):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model_fp32(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

print("\n[Original Model] Accuracy after tuning:")
print(f"Accuracy: {evaluate(model_fp32, test_loader):.2f}%")

# ========== PTQ ==========
ptq_model = mobilenet_v2(pretrained=True, quantize=False)
ptq_model.classifier[1] = nn.Linear(ptq_model.last_channel, 10)
ptq_model.load_state_dict(model_fp32.state_dict())
ptq_model.eval()
ptq_model.fuse_model()
ptq_model.qconfig = torch.quantization.get_default_qconfig('fbgemm')
torch.quantization.prepare(ptq_model, inplace=True)

print("\n[PTQ Model] Calibrating observers...")
ptq_model.cpu()  # 양자화 전환 전에는 CPU여야 함
with torch.no_grad():
    for i, (inputs, _) in enumerate(train_loader):
        ptq_model(inputs.cpu())
        if i > 100:
            break

ptq_model = torch.quantization.convert(ptq_model, inplace=False)
print("\n[PTQ Model] Accuracy:")
print(f"Accuracy: {evaluate(ptq_model, test_loader):.2f}%")

# ========== QAT ==========
qat_model = mobilenet_v2(pretrained=True, quantize=False)
qat_model.classifier[1] = nn.Linear(qat_model.last_channel, 10)
qat_model.load_state_dict(model_fp32.state_dict())
qat_model.train()
qat_model.fuse_model()
qat_model.qconfig = torch.quantization.get_default_qat_qconfig('fbgemm')
torch.quantization.prepare_qat(qat_model, inplace=True)
qat_model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(qat_model.parameters(), lr=0.001, momentum=0.9)

print("\n[QAT Model] Training...")
for epoch in range(5):
    qat_model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = qat_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

qat_model.eval()
qat_model = torch.quantization.convert(qat_model.cpu(), inplace=False)
print("\n[QAT Model] Accuracy after tuning:")
print(f"Accuracy: {evaluate(qat_model, test_loader):.2f}%")


torce device : cuda


C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



[Original Model] Training full model...
Epoch 1, Loss: 0.4621
Epoch 2, Loss: 0.2177
Epoch 3, Loss: 0.1435
Epoch 4, Loss: 0.0982
Epoch 5, Loss: 0.0723
Epoch 6, Loss: 0.0553
Epoch 7, Loss: 0.0387
Epoch 8, Loss: 0.0331
Epoch 9, Loss: 0.0253
Epoch 10, Loss: 0.0219

[Original Model] Accuracy after tuning:
Accuracy: 94.35%

[PTQ Model] Calibrating observers...


C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\ao\quantization\observer.py:244: UserWarning: Please use quant_min and quant_max to specify the range for observers.                     reduce_range will be deprecated in a future release of PyTorch.
  warnings.warn(
C:\Users\whitedk\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\ao\quantization\observer.py:1333: UserWarning: must run observer before calling calculate_qparams.                                    Returning default scale and zero point 
  warnings.warn(



[PTQ Model] Accuracy:
